In [ ]:
def process_batch(df, epoch_id):
    from pyspark.sql.functions import expr

    # Gom 5 ảnh mỗi lần
    batched_df = df.withColumn("row_id", expr("monotonically_increasing_id()")) \
                   .withColumn("group_id", (col("row_id") / 5).cast("int")) \
                   .groupBy("group_id") \
                   .agg(f.collect_list("data").alias("batch_data")) \
                   .select(to_json(col("batch_data")).alias("value"))

    # Ghi vào Kafka
    batched_df.write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("topic", "check_error") \
        .save()

parsed_df.writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", "./checkpoints/check_error") \
    .start()


array([2.23606798, 5.        ])

In [8]:
a = [1, 2]
a.append([1, 2, [1, 2]])
a

[1, 2, [1, 2, [1, 2]]]

In [6]:
import numpy as np
import pandas as pd

temp = {
    'class_ID' : [0, 0, 0, 0],
    'color_feature' : [[1,1,1],
                       [2,2,2],
                       [3,3,3],
                       [4,4,4]]
}
# temp = pd.DataFrame(temp)
# np.linalg.norm(temp['color_feature'].tolist(), axis = 1)
for char in temp.values():
    print(char)

[0, 0, 0, 0]
[[1, 1, 1], [2, 2, 2], [3, 3, 3], [4, 4, 4]]


In [18]:
pd.DataFrame(np.array(temp['color_feature'].values.tolist()).reshape(-1, 3))

,0,1,2
0,1,1,1
1,2,2,2
2,3,3,3
3,4,4,4


In [20]:
(temp['class_ID'] + 1).values

array([1, 1, 1, 1])

In [33]:
import numpy as np

a = np.array([1, 2])
b = np.array([3, 4])

1 / (a + b)

array([0.25      , 0.16666667])

In [1]:
bool([])

False

In [50]:
box_1 = [1, 1, 2, 2]

box = {
    'class_ID' : [0, 0, 0, 0],
    'color_feature' : [[1,1,1],
                       [2,2,2],
                       [3,3,3],
                       [4,4,4]],
    'box' : [[1,1,1,1],
            [2,2,2,2],
            [3,3,3,3],
            [4,4,4,4]]
}
# del box['class_ID']
box = np.array(pd.DataFrame(box)['box'].values.tolist()).reshape(-1, 4)
box = pd.DataFrame(box, columns = ['x', 'y', 'h', 'w'])

box['x_min'] = (box['x'] - box['h'] / 2).clip(lower=box_1[0])
box['y_min'] = (box['y'] - box['w'] / 2).clip(lower=box_1[1])
box['x_max'] = (box['x'] + box['h'] / 2).clip(upper=box_1[2])
box['y_max'] = (box['y'] + box['w'] / 2).clip(upper=box_1[3])

box['area'] = box['h'] * box['w']
box['inter_h'] = (box['x_max'] - box['x_min']).clip(lower=0)
box['inter_w'] = (box['y_max'] - box['y_min']).clip(lower=0)

area_match = (box['inter_w'] * box['inter_h']).values
area_total = box['area'].values + 2 - area_match
IOU = area_match / area_total
temp_1 = np.array([i >= 0.7 for i in IOU])
temp_1

array([False, False, False, False])

In [51]:
box = {
    'class_ID' : [0, 0, 0, 0],
    'color_feature' : [[1,1,1],
                       [2,2,2],
                       [3,3,3],
                       [4,4,4]],
    'box' : [[1,1,1,1],
            [2,2,2,2],
            [3,3,3,3],
            [4,4,4,4]]
}
box = pd.DataFrame(box)
data = box['color_feature'].tolist()

vec1 = np.array([1, 2, 3])
vec2 = np.array(data)

dot = np.dot(vec2, vec1)
norm1 = np.linalg.norm(vec1)
norm2 = np.linalg.norm(vec2, axis=1)

print(np.where(norm1 * norm2 != 0, dot / (norm1 * norm2), 0))
print()
temp_2 = np.where(norm1 * norm2 != 0, dot / (norm1 * norm2) >= 0.7, False)
print(temp_2)


[0.9258201 0.9258201 0.9258201 0.9258201]

[ True  True  True  True]


In [ ]:
temp_1 & temp_2

array([False, False, False, False])

In [60]:
box = {
    'class_ID' : [0, 0, 0, 0],
    'color_feature' : [[1,1,1],
                       [2,2,2],
                       [3,3,3],
                       [4,4,4]],
    'box' : [[1,1,1,1],
            [2,2,2,2],
            [3,3,3,3],
            [4,4,4,4]]
}
print(pd.DataFrame(box)[[True, False, True, False]]['color_feature'].tolist())
print(pd.DataFrame(box)[[True, False, True, False]].reset_index(drop = True))

[[1, 1, 1], [3, 3, 3]]
   class_ID color_feature           box
0         0     [1, 1, 1]  [1, 1, 1, 1]
1         0     [3, 3, 3]  [3, 3, 3, 3]


# temp

In [29]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Check:
    Active: bool = False
    Kind: int = 0
    List_Vehicle: List[int] = field(default_factory=lambda: [])
    Angle: List[float] = field(default_factory=lambda: [])

    def __add__(self, other):
        if not isinstance(other, Check):
            return NotImplemented
        return Check(
            Active=self.Active,
            Kind=self.Kind,
            List_Vehicle = self.List_Vehicle + other.List_Vehicle,
            Angle = self.Angle + other.Angle
            )

    def __repr__(self):
        return f"Check({self.Active}, {self.Kind}, {self.List_Vehicle}, {self.Angle})"

## temp1

In [32]:
import numpy as np

# Tạo lưới 2x3 check
row, col = 2, 3
grid = np.array([[Check() for _ in range(col)] for _ in range(row)], dtype=object)

# Tạo check để cộng vào
c = Check(True, 2, [1], [10])

grid[:, 1] += c
grid

array([[Check(False, 0, [], []), Check(False, 0, [1], [10]),
        Check(False, 0, [], [])],
       [Check(False, 0, [], []), Check(False, 0, [1], [10]),
        Check(False, 0, [], [])]], dtype=object)

In [26]:
grid = np.array([[[0, 0] for _ in range(col)] for _ in range(row)], dtype=object)
grid[1, :, 1] = 1
grid

array([[[0, 0],
        [0, 0],
        [0, 0]],

       [[0, 1],
        [0, 1],
        [0, 1]]], dtype=object)

In [2]:
import os
print(os.listdir())

['Backend.py', 'Charater.py', 'Checker.py', 'DeepSort.py', 'Def_common.py', 'Draw.py', 'FrontEnd.py', 'Map.py', 'Process_OutPut.py', 'test.ipynb', '__pycache__']


In [3]:
from ultralytics import YOLO
import cv2

# Load mô hình .pt
model = YOLO("./final.pt")

img = cv2.imread('./img_1.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

result = model(img)


0: 384x640 4 xe_mays, 1 o_to, 77.8ms
Speed: 4.3ms preprocess, 77.8ms inference, 5.6ms postprocess per image at shape (1, 3, 384, 640)


In [15]:
result[0].boxes[0].xywh[0]

tensor([306.7487, 247.8339,  21.0904,  42.5434])

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("YOLOBroadcast").getOrCreate()
sc = spark.sparkContext

# Broadcast trọng số
# broadcast_weights = sc.broadcast(weights)


In [ ]:
from ultralytics import YOLO
import torch

def run_inference_on_partition(partition):
    # Load mô hình YOLO trống (cùng loại với best.pt)
    model = YOLO("yolov8n.pt")  # hoặc đúng backbone bạn đang dùng
    model.model.load_state_dict(broadcast_weights.value)
    model.eval()

    for record in partition:
        # Giả sử record là đường dẫn ảnh
        results = model(record)
        yield results[0].tojson()  # hoặc .boxes, .probs tuỳ bạn


In [44]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Char:
    class_id: int = 0
    HSV: List[int] = field(default_factory=lambda: [])
    box: List[int] = field(default_factory=lambda: [])

In [61]:
def CheckColorFeature(self, data, ratio, return_score = False):
    vec1 = np.array(self.Color_Feature)
    vec2 = np.array(data)

    dot = np.dot(vec2, vec1)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2, axis=1)
    
    if return_score:
        temp = np.where(norm1 * norm2 != 0, dot / (norm1 * norm2), 0)
        return temp
    return np.where(norm1 * norm2 != 0, dot / (norm1 * norm2) >= ratio, False)

In [65]:
import pandas as pd

box = {
    'class_ID' : [0],
    'color_feature' : [[1,1,1]],
    'box' : [[1,1,1,1]]
}
print(pd.DataFrame(box)['color_feature'].values.tolist()[0])

[1, 1, 1]


In [ ]:
# Chạy file .py bằng cmd và ẩn cửa sổ cmd

import subprocess
import sys
import os

def run_python_script():
    # Đường dẫn đến file script.py (có thể thay đổi theo vị trí của bạn)
    script_path = "C:\\path\\to\\your\\script.py"
    
    # Sử dụng pythonw.exe để chạy script mà không hiển thị cửa sổ CMD
    subprocess.Popen([sys.executable.replace('python.exe', 'pythonw.exe'), script_path])

# Chạy hàm
run_python_script()


In [ ]:
import tkinter as tk
from PIL import Image, ImageTk

root = tk.Tk()
root.title("Hiển thị ảnh với Pillow")

# Mở ảnh bằng PIL
image = Image.open("path/to/image.jpg")
# Resize nếu muốn (tuỳ chọn)
image = image.resize((400, 300))  # chiều rộng x chiều cao

# Chuyển thành ảnh dùng được cho Tkinter
tk_image = ImageTk.PhotoImage(image)

# Gắn ảnh vào Label
label = tk.Label(root, image=tk_image)
label.pack()

root.mainloop()


In [ ]:
# hiện ảnh từ ndarray

import cv2
import numpy as np
from PIL import Image, ImageTk
import tkinter as tk

# Đọc ảnh bằng OpenCV (ảnh dạng ndarray)
cv_img = cv2.imread("path/to/image.jpg")  # ảnh ở dạng BGR
cv_img = cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB)  # Chuyển sang RGB

# Chuyển ndarray (OpenCV) -> PIL.Image
pil_img = Image.fromarray(cv_img)

# Chuyển PIL.Image -> ImageTk.PhotoImage để dùng trong Tkinter
tk_img = ImageTk.PhotoImage(pil_img)

# Tạo giao diện Tkinter
root = tk.Tk()
label = tk.Label(root, image=tk_img)
label.pack()

root.mainloop()


In [ ]:
import tkinter as tk
from visual.kafkaController import Kafka_Controller

kafka = Kafka_Controller()


class Application:
    def __init__(self, root):
        self.root = root
        self.root.title("Menu Navigation Example")
        self.root.geometry("600x400")
        
        # Tạo Menu Bar
        menubar = tk.Menu(self.root)
        self.root.config(menu=menubar)

        # Tạo các menu chính
        menubar.add_command(label="Page 1", command=self.show_page1)
        menubar.add_command(label="Page 2", command=self.show_page2)
        menubar.add_command(label="Page 3", command=self.show_page3)
        
        # Tạo các Frame cho các trang
        self.page1_frame = tk.Frame(self.root)
        self.page2_frame = tk.Frame(self.root)
        self.page3_frame = tk.Frame(self.root)
        
        self.create_page1()
        self.create_page2()
        self.create_page3()

        # Mặc định hiển thị Page 1 khi mở ứng dụng
        self.show_page1()

    def create_page1(self):
        button = tk.Button(self.page1_frame, text="Run Kafka_Server", command=kafka.Start_Kafka_Server)
        button.pack(pady=20)
        label = tk.Label(self.page1_frame, text="This is Page 1", font=("Arial", 20))
        label.pack(pady=20)
        button = tk.Button(self.page1_frame, text="Go to Page 2", command=self.show_page2)
        button.pack(pady=10)

    def create_page2(self):
        label = tk.Label(self.page2_frame, text="This is Page 2", font=("Arial", 20))
        label.pack(pady=20)
        button = tk.Button(self.page2_frame, text="Go to Page 3", command=self.show_page3)
        button.pack(pady=10)

    def create_page3(self):
        label = tk.Label(self.page3_frame, text="This is Page 3", font=("Arial", 20))
        label.pack(pady=20)
        button = tk.Button(self.page3_frame, text="Go to Page 1", command=self.show_page1)
        button.pack(pady=10)

    def show_page1(self):
        # Ẩn tất cả các trang
        self.page2_frame.pack_forget()
        self.page3_frame.pack_forget()
        # Hiển thị trang 1
        self.page1_frame.pack(fill="both", expand=True)

    def show_page2(self):
        # Ẩn tất cả các trang
        self.page1_frame.pack_forget()
        self.page3_frame.pack_forget()
        # Hiển thị trang 2
        self.page2_frame.pack(fill="both", expand=True)

    def show_page3(self):
        # Ẩn tất cả các trang
        self.page1_frame.pack_forget()
        self.page2_frame.pack_forget()
        # Hiển thị trang 3
        self.page3_frame.pack(fill="both", expand=True)


root = tk.Tk()
root.title("Chạy CMD từ nút")
root.geometry("1024x720")


app = Application(root)

root.mainloop()

In [ ]:
import tkinter as tk
from tkinter import filedialog

def chon_video():
    file_path = filedialog.askopenfilename(
        title="Chọn file video",
        filetypes=[("Video files", "*.mp4 *.avi *.mkv *.mov *.wmv"), ("All files", "*.*")]
    )
    if file_path:
        print("Đã chọn video:", file_path)

# Tạo cửa sổ đơn giản
root = tk.Tk()
root.title("Chọn file video")
root.geometry("300x150")

btn = tk.Button(root, text="Chọn Video", command=chon_video)
btn.pack(pady=40)

root.mainloop()


In [ ]:
import tkinter as tk

class Page1(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent)
        label = tk.Label(self, text="Đây là trang 1", font=("Arial", 20))
        label.pack(pady=20)
        button = tk.Button(self, text="Đến trang 2", command=lambda: controller.show_frame("Page2"))
        button.pack()

class Page2(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent)
        label = tk.Label(self, text="Đây là trang 2", font=("Arial", 20))
        label.pack(pady=20)
        button = tk.Button(self, text="Quay lại trang 1", command=lambda: controller.show_frame("Page1"))
        button.pack()

class MainApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Ứng dụng nhiều trang")
        self.geometry("600x400")

        self.frames = {}

        # Container để giữ các frame
        container = tk.Frame(self)
        container.pack(fill="both", expand=True)

        # Đăng ký các trang vào frames
        for F in (Page1, Page2):
            page_name = F.__name__
            frame = F(parent=container, controller=self)
            self.frames[page_name] = frame
            frame.grid(row=0, column=0, sticky="nsew")

        self.show_frame("Page1")

    def show_frame(self, page_name):
        frame = self.frames[page_name]
        frame.tkraise()  # Đưa frame này lên trên

if __name__ == "__main__":
    app = MainApp()
    app.mainloop()


In [11]:
import subprocess

class KafkaUtils:
    @staticmethod
    def Create_Topic(data):
        # Tách dữ liệu để lấy topic và cổng
        temp = data.split("-")
        topic = temp[0]
        port = temp[1]
        
        # Kiểm tra lại nếu có phần tên topic và port
        if not topic or not port:
            print("Dữ liệu không hợp lệ!")
            return
        
        # Lệnh để tạo topic
        command = [
            'C:\\kafka_2.13-4.0.0\\bin\\windows\\kafka-topics.bat',
            '--create',
            '--topic', topic,
            '--bootstrap-server', f'localhost:{port}',
            # '--partitions', '1',
            # '--replication-factor', '1'
        ]
        
        # In ra command để kiểm tra xem lệnh có đúng không
        print("Running command:", " ".join(command))
        
        # Thực thi lệnh và thu thập kết quả
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        
        # In kết quả trả về hoặc lỗi
        print("Output:", result.stdout)
        print("Error:", result.stderr)

# Ví dụ gọi hàm
KafkaUtils.Create_Topic("lksdjfkldsjf-9092")


Running command: C:\kafka_2.13-4.0.0\bin\windows\kafka-topics.bat --create --topic lksdjfkldsjf --bootstrap-server localhost:9092
Output: 2025-05-11T02:43:53.770947300Z main ERROR Reconfiguration failed: No configuration found for '5a07e868' at 'null' in 'null'
Created topic lksdjfkldsjf.

Error: 


In [5]:
command = [
            'C:\\kafka_2.13-4.0.0\\bin\\windows\\kafka-topics.bat',
            '--list',
            '--bootstrap-server',
            f'localhost:9092',
        ]
        
result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
result.stdout

"2025-05-11T02:36:36.128294800Z main ERROR Reconfiguration failed: No configuration found for '5a07e868' at 'null' in 'null'\n__consumer_offsets\nimage\nmy_topic\nmytopic\ntiktok\n"

In [12]:
import subprocess

def delete_topic(topic, port):
    command = [
        'C:\\kafka_2.13-4.0.0\\bin\\windows\\kafka-topics.bat',
        '--delete',
        '--topic', topic,
        '--bootstrap-server', f'localhost:{port}'
    ]
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print(result.stdout)
    print(result.stderr)

# Gọi thử
delete_topic('my_topic', 9092)


2025-05-11T02:45:53.196216400Z main ERROR Reconfiguration failed: No configuration found for '5a07e868' at 'null' in 'null'




In [4]:
import subprocess

subprocess.run(command = ["C:\kafka_2.13-4.0.0"])

TypeError: Popen.__init__() got an unexpected keyword argument 'command'

In [1]:
import os
print(os.environ.get('PYSPARK_PYTHON'))

C:\Users\trung\AppData\Local\Programs\Python\Python310\python.exe


In [5]:
temp = []
temp.extend([[1, 3, 4], [12]])
temp

[[1, 3, 4], [12]]

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, StringType
import pandas as pd

# Khởi tạo SparkSession
spark = SparkSession.builder.master("local").appName("PandasUDFExample").getOrCreate()

# Định nghĩa UDF: Chia chuỗi thành mảng các từ
@pandas_udf(ArrayType(StringType()))
def split_string_to_array(s: pd.Series) -> pd.Series:
    return s.str.split(" ")

# Tạo DataFrame mẫu
data = [("Hello world",), ("Spark is great",), ("Pandas UDF",)]
df = spark.createDataFrame(data, ["text"])

# Áp dụng UDF để chuyển chuỗi thành mảng
df_with_array = df.withColumn("words", split_string_to_array(df["text"]))

# Hiển thị kết quả
# df_with_array.show(truncate=False)

In [4]:
df_with_array.show(5)

Py4JJavaError: An error occurred while calling o97.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 2.0 failed 1 times, most recent failure: Lost task 0.0 in stage 2.0 (TID 2) (VNxVTTr executor driver): java.lang.UnsupportedOperationException: sun.misc.Unsafe or java.nio.DirectByteBuffer.<init>(long, int) not available
	at org.apache.arrow.memory.util.MemoryUtil.directBuffer(MemoryUtil.java:174)
	at org.apache.arrow.memory.ArrowBuf.getDirectBuffer(ArrowBuf.java:229)
	at org.apache.arrow.memory.ArrowBuf.nioBuffer(ArrowBuf.java:224)
	at org.apache.arrow.vector.ipc.WriteChannel.write(WriteChannel.java:133)
	at org.apache.arrow.vector.ipc.message.MessageSerializer.writeBatchBuffers(MessageSerializer.java:303)
	at org.apache.arrow.vector.ipc.message.MessageSerializer.serialize(MessageSerializer.java:276)
	at org.apache.arrow.vector.ipc.ArrowWriter.writeRecordBatch(ArrowWriter.java:147)
	at org.apache.arrow.vector.ipc.ArrowWriter.writeB